In [16]:
import re

# Simple word lists
positive_words = ["good", "great", "excellent", "amazing", "wonderful", "love", "fantastic", "best", "enjoyed"]
negative_words = ["bad", "terrible", "awful", "worst", "boring", "hate", "poor", "disappointing", "sad"]

In [17]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

In [18]:
def predict_sentiment(review):
    tokens = preprocess(review)
    pos_count = sum(word in positive_words for word in tokens)
    neg_count = sum(word in negative_words for word in tokens)

    if pos_count > neg_count:
        return "Positive Review"
    elif neg_count > pos_count:
        return "Negative Review"
    else:
        return "Neutral / Mixed Review"

In [19]:
if __name__ == "__main__":
    review = input("Enter your movie review: ").strip()
    if review:
        result = predict_sentiment(review)
        print(result)
    else:
        print("Please enter a review first.")

Positive Review


In [20]:
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
df=pd.read_csv(r"c:\DATA SCIENCE\Machine_Learning\Datasets\data.csv")
df.head()


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


In [21]:
df.isnull().sum()

Sentence     0
Sentiment    0
dtype: int64

In [22]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

#text preprocessing---->stemming 
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prasa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
corpus=[]
for i in range(0,len(df)):
    review=re.sub('[^a-zA-Z]',' ',df['Sentence'][i])
    review=review.lower()
    review=review.split()
    review=[stemmer.stem(word) for word in review if not word in stopwords.words('english')]
    review=' '.join(review)
    corpus.append(review)


In [24]:
df.columns

Index(['Sentence', 'Sentiment'], dtype='object')

In [25]:
df["Sentiment"].value_counts()

Sentiment
neutral     3130
positive    1852
negative     860
Name: count, dtype: int64

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Sentiment"] = le.fit_transform(df["Sentiment"])
df.head()

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,2
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",0
2,"For the last quarter of 2010 , Componenta 's n...",2
3,According to the Finnish-Russian Chamber of Co...,1
4,The Swedish buyout firm has sold its remaining...,1


In [38]:
import pickle
with open("labelencoder.pkl", "wb") as f: 
    pickle.dump(le, f)

In [27]:
x=df["Sentence"].values
y=df["Sentiment"].values

In [ ]:
# Train/test split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)


In [ ]:
vectorizer = TfidfVectorizer(analyzer=lambda x: x, stop_words="english")
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)


c:\Users\prasa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


In [39]:
with open("vectorizer.pkl", "wb") as f: 
    pickle.dump(vectorizer, f)

In [ ]:
# Train model
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [40]:
with open("model.pkl", "wb") as f: 
    pickle.dump(model, f)

In [ ]:
# Prediction function
def predict_sentiment(review): 
    processed = preprocess(review) 
    vectorized = vectorizer.transform([processed]) 
    prediction = model.predict(vectorized)[0] 
    probability = model.predict_proba(vectorized)[0] 
    confidence = max(probability) * 100 
    return "negative" if prediction == 1 else "positive", confidence


In [32]:
if __name__ == "__main__": 
    review = input("Enter your movie review: ").strip() 
    if review: 
        sentiment, confidence = predict_sentiment(review) 
        print(sentiment, f"(Confidence: {confidence:.2f}%)") 
    else: 
        print("⚠ Please enter a review first.")

negative (Confidence: 43.52%)


In [33]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess(text):
    # Remove non-alphabetic characters
    review = re.sub('[^a-zA-Z]', ' ', text)
    # Lowercase
    review = review.lower()
    # Tokenize
    words = review.split()
    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)


In [34]:
df["clean_text"] = df["Sentence"].apply(preprocess)

x = df["clean_text"].values
y = df["Sentiment"].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)  # limit features for efficiency
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)


In [35]:
model = LogisticRegression(
    max_iter=2000,
    C=2,                # less regularization → better fit
    solver="liblinear", # good for small datasets
    class_weight="balanced" # handles imbalanced data
)
model.fit(x_train, y_train)


c:\Users\prasa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,2
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'liblinear'
,max_iter,2000
,multi_class,'deprecated'


In [36]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7082976903336184
              precision    recall  f1-score   support

           0       0.40      0.40      0.40       175
           1       0.76      0.80      0.78       622
           2       0.77      0.70      0.74       372

    accuracy                           0.71      1169
   macro avg       0.64      0.63      0.64      1169
weighted avg       0.71      0.71      0.71      1169



In [37]:
if __name__ == "__main__":
    review = input("Enter your movie review: ").strip()
    if review:
        sentiment, confidence = predict_sentiment(review)
        print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.2f}%)")
    else:
        print("⚠ Please enter a review first.")


Predicted Sentiment: positive (Confidence: 37.81%)
